# BarqTrain Colab: Benchmark Comparison

This notebook compares:
- Baseline Hugging Face/PyTorch
- BarqTrain patched model
- BarqTrain patched model with CUDA path enabled
- Unsloth AI (when available)

Metrics:
- Inference latency
- Tokens/sec
- Peak VRAM
                


In [ ]:
%%bash
set -e
pip -q install --upgrade pip
pip -q install torch transformers datasets accelerate peft bitsandbytes pandas
pip -q install unsloth || true
if [ ! -d /content/BarqTrain ]; then
  git clone https://github.com/YASSERRMD/BarqTrain.git /content/BarqTrain
fi
cd /content/BarqTrain
pip -q install -e .
                


In [ ]:
%cd /content/BarqTrain

import os
import time
import torch
import pandas as pd

from transformers import AutoModelForCausalLM, AutoTokenizer
from barqtrain import patch_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
PROMPT = "Explain why fused kernels can improve LLM training throughput."
NEW_TOKENS = 64
RUNS = 5

print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Warning: CUDA not available. GPU benchmarks will be limited.')
                


In [ ]:
def _to_device(batch, device):
    if device == "cuda":
        return {k: v.cuda() for k, v in batch.items()}
    return batch


def benchmark_generate(model, tokenizer, prompt, new_tokens=64, runs=5):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = _to_device(inputs, DEVICE)

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    # warmup
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=16)

    latencies = []
    for _ in range(runs):
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=new_tokens)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        latencies.append(time.time() - t0)

    elapsed = sum(latencies) / len(latencies)
    generated_tokens = out.shape[-1] - inputs["input_ids"].shape[-1]
    tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
    peak_vram_mb = (torch.cuda.max_memory_allocated() / 1024**2) if DEVICE == "cuda" else 0.0

    return {
        "avg_latency_s": elapsed,
        "tokens_per_sec": tokens_per_sec,
        "peak_vram_mb": peak_vram_mb,
    }
                


In [ ]:
results = []

dtype = torch.float16 if DEVICE == "cuda" else torch.float32

# 1) Baseline
baseline_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if baseline_tok.pad_token is None:
    baseline_tok.pad_token = baseline_tok.eos_token
baseline_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, trust_remote_code=True)
if DEVICE == "cuda":
    baseline_model = baseline_model.cuda()

m = benchmark_generate(baseline_model, baseline_tok, PROMPT, NEW_TOKENS, RUNS)
results.append({"setup": "baseline_hf", **m})

# 2) BarqTrain patch (default runtime behavior)
patched_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if patched_tok.pad_token is None:
    patched_tok.pad_token = patched_tok.eos_token
patched_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, trust_remote_code=True)
if DEVICE == "cuda":
    patched_model = patched_model.cuda()
patch_model(patched_model)

m = benchmark_generate(patched_model, patched_tok, PROMPT, NEW_TOKENS, RUNS)
results.append({"setup": "barqtrain_patch", **m})

# 3) BarqTrain patch with CUDA auto-build explicitly enabled
os.environ["BARQTRAIN_AUTO_BUILD"] = "1"
os.environ["BARQTRAIN_VERBOSE_BUILD"] = "0"
patched_cuda_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if patched_cuda_tok.pad_token is None:
    patched_cuda_tok.pad_token = patched_cuda_tok.eos_token
patched_cuda_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, trust_remote_code=True)
if DEVICE == "cuda":
    patched_cuda_model = patched_cuda_model.cuda()
patch_model(patched_cuda_model)

m = benchmark_generate(patched_cuda_model, patched_cuda_tok, PROMPT, NEW_TOKENS, RUNS)
results.append({"setup": "barqtrain_patch_cuda_enabled", **m})
                


In [ ]:
# 4) Unsloth AI (optional)
try:
    from unsloth import FastLanguageModel

    unsloth_model, unsloth_tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=1024,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(unsloth_model)

    if DEVICE == "cuda":
        unsloth_model = unsloth_model.cuda()

    m = benchmark_generate(unsloth_model, unsloth_tok, PROMPT, NEW_TOKENS, RUNS)
    results.append({"setup": "unsloth_ai", **m})
except Exception as e:
    print("Unsloth benchmark skipped:", repr(e))
                


In [ ]:
df = pd.DataFrame(results)
df = df.sort_values("tokens_per_sec", ascending=False).reset_index(drop=True)
df
                


In [ ]:
out_path = "/content/barqtrain_benchmark_results.csv"
df.to_csv(out_path, index=False)
print('Saved:', out_path)
                
